## HYPER PARAMETER TUNING

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load data
df = pd.read_csv('paysim_balanced_clean.csv')

# Encode transaction type
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

# Features & target
X = df[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig']]
y = df['isFraud']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Hyperparameter Tuning
params = {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}
grid = GridSearchCV(LogisticRegression(class_weight='balanced'), params, cv=5, scoring='f1')
grid.fit(X_train_sc, y_train)

print("Best params:", grid.best_params_)
print("Best F1 score:", grid.best_score_)

Best params: {'C': 10, 'solver': 'lbfgs'}
Best F1 score: 0.885011468561921


## NAIVE BAYES TUNING

In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV

# Naive Bayes hyperparameter tuning
nb_params = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]}
nb_grid = GridSearchCV(GaussianNB(), nb_params, cv=5, scoring='f1')
nb_grid.fit(X_train_sc, y_train)

print("Best NB params:", nb_grid.best_params_)
print("Best NB F1 score:", nb_grid.best_score_)

Best NB params: {'var_smoothing': 1e-09}
Best NB F1 score: 0.501020970810416


## THREE_MODELS_TUNING(Random Forest + XGBoost + SVM)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import joblib, json

# Load data
df = pd.read_csv('paysim_balanced_clean.csv')
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])

X = df[['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig']]
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Random Forest tuning
rf_params = {'n_estimators': [50, 100], 'max_depth': [5, 10]}
rf_grid = GridSearchCV(RandomForestClassifier(class_weight='balanced'), rf_params, cv=3, scoring='f1')
rf_grid.fit(X_train_sc, y_train)
print("RF Best:", rf_grid.best_params_, "F1:", round(rf_grid.best_score_, 3))
joblib.dump(rf_grid.best_estimator_, 'rf_model.pkl')

# XGBoost tuning
xgb_params = {'n_estimators': [50, 100], 'max_depth': [3, 5], 'learning_rate': [0.1, 0.3]}
xgb_grid = GridSearchCV(XGBClassifier(eval_metric='logloss'), xgb_params, cv=3, scoring='f1')
xgb_grid.fit(X_train_sc, y_train)
print("XGB Best:", xgb_grid.best_params_, "F1:", round(xgb_grid.best_score_, 3))
joblib.dump(xgb_grid.best_estimator_, 'xgb_model.pkl')

# SVM tuning
svm_params = {'C': [0.1, 1], 'kernel': ['rbf', 'linear']}
svm_grid = GridSearchCV(SVC(class_weight='balanced', probability=True), svm_params, cv=3, scoring='f1')
svm_grid.fit(X_train_sc, y_train)
print("SVM Best:", svm_grid.best_params_, "F1:", round(svm_grid.best_score_, 3))
joblib.dump(svm_grid.best_estimator_, 'svm_model.pkl')

print("All 3 models saved!")

RF Best: {'max_depth': 10, 'n_estimators': 100} F1: 0.992
XGB Best: {'learning_rate': 0.3, 'max_depth': 5, 'n_estimators': 100} F1: 0.993
SVM Best: {'C': 1, 'kernel': 'rbf'} F1: 0.884
All 3 models saved!
